---
## Section 1: Tokenization

### Why This Matters

An LLM **never sees your text**. It sees a sequence of integer IDs called **tokens**.  
Tokens are subword units — not characters, not words, but chunks derived from training data frequency.

**3 things tokenization affects directly:**
1. **Context window usage** — you're charged in tokens, not words
2. **Model behavior on technical terms** — a term split into 4 tokens behaves differently than 1
3. **Format compliance** — delimiters that tokenize awkwardly cause malformed structured output

---

In [ ]:
# ─────────────────────────────────────────────────────────────
# Basic tokenization
#
# tiktoken encodes text → list of token IDs
# Different models use different vocabularies (encodings)
# gpt-4o uses "o200k_base" encoding
# ─────────────────────────────────────────────────────────────

# Load the encoder for GPT-4o
enc = tiktoken.get_encoding("o200k_base")  # Used by GPT-4o family

# Our example text
sample_text = "The transformer architecture underlies every major LLM today."

# Encode: text → list of integer token IDs
token_ids = enc.encode(sample_text)

# Decode each token ID back to its string for human inspection
token_strings = [enc.decode([t]) for t in token_ids]

print(f"Original text:\n  {sample_text}")
print(f"\nToken count: {len(token_ids)}")
print(f"\nToken IDs:   {token_ids}")
print(f"\nToken strings (what the model 'sees' as chunks):")
for i, (tid, tstr) in enumerate(zip(token_ids, token_strings)):
    print(f"  [{i:2d}] ID={tid:6d}  →  '{tstr}'")

In [ ]:
# ─────────────────────────────────────────────────────────────
# HELPER: Reusable tokenization inspection function
# Use this throughout the notebook to inspect any text
# ─────────────────────────────────────────────────────────────

def inspect_tokens(text: str, encoding_name: str = "o200k_base") -> dict:
    """
    Tokenizes `text` and prints a human-readable breakdown.
    Returns a dict with token_ids, token_strings, and count.
    """
    enc = tiktoken.get_encoding(encoding_name)
    ids = enc.encode(text)
    strings = [enc.decode([t]) for t in ids]
    
    print(f"Text: {repr(text)}")
    print(f"Token count: {len(ids)}")
    print(f"Tokens: {strings}")
    print("-" * 50)
    
    return {"ids": ids, "strings": strings, "count": len(ids)}


# ─────────────────────────────────────────────────────────────
# DEMO: Notice how the same concept tokenizes differently
# depending on casing, spacing, and hyphenation
# ─────────────────────────────────────────────────────────────

examples = [
    "transformer",
    "Transformer",
    "TRANSFORMER",
    "transformer architecture",
    "unrecognizable",   # longer word — watch how it splits
    "the",             # very common word — likely 1 token
    "McKinsey",        # proper noun — how does it split?
    "JSON",
    "{",               # delimiter character — important for structured output!
]

for text in examples:
    inspect_tokens(text)

In [ ]:
# ─────────────────────────────────────────────────────────────
# TODO 1 — Token Count Estimation (5 min)
#
# In production, you need to estimate token costs BEFORE
# sending a request. This function does that.
#
# TASK:
#   1. Count the tokens in the system prompt below
#   2. Count the tokens in the user message below
#   3. Estimate the TOTAL context consumed (system + user)
#   4. Add your own example — try a prompt you'd use at work
# ─────────────────────────────────────────────────────────────

system_prompt = """You are a technical document analyzer. 
Extract the key findings from the document and return them as a JSON array. 
Each finding should have: section, finding, confidence (high/medium/low)."""

user_message = """Please analyze the following contract clause:
The Vendor shall provide a 99.9% uptime SLA for all production systems, 
measured monthly, excluding scheduled maintenance windows not exceeding 4 hours per quarter."""

# --- YOUR CODE HERE ---
# Hint: Use the inspect_tokens() function defined above
# or use enc.encode() directly

# system_tokens = ???
# user_tokens = ???
# total = ???
# print(f"System prompt: {system_tokens} tokens")
# print(f"User message:  {user_tokens} tokens")
# print(f"Total context: {total} tokens")
# print(f"Budget remaining (128k window): {128000 - total} tokens")